# Arabic news summarization — one FP32 experiment on Colab T4

Choose **Runtime → Disconnect and delete runtime**, then connect a **T4 GPU** runtime.
Run cells **1–7 in order** (or Run all). The setup asks you to mount Drive and enter
the free storage shown at https://drive.google.com/drive/u/0/quota (no quota API).
Edit the settings in cell 1 before starting. Cell 7 trains and immediately exports;
cells 8–10 evaluate the safely saved model, show examples, and accept custom text.
Cell 11 optionally launches the existing Arabic Gradio app.

One full-parameter, teacher-forced experiment: `UBC-NLP/AraT5v2-base-1024`, Arabic
`csebuetnlp/xlsum`, LR **5e-5**, **FP32**. No second experiment or learning-rate sweep.
The first three optimizer steps of this same run measure GPU speed and print an ETA.
The default is **200 optimizer steps**, accumulation 8, with a **90-minute training
budget**; the budget stops gracefully at a step boundary and then saves normally.
Reserve additional session time for downloads and the final Drive upload.
Step count and runtime do **not** guarantee summary quality.

**No periodic checkpoints, optimizer/scheduler/RNG saves, or disconnect recovery.**
Code, caches, data and Trainer output stay under `/content`. Only the final
weights/tokenizer/inference configuration go to a new timestamped Drive directory.
`full-v2-fp32-final-01` and all previous runs/Trash are untouched. A disconnect before
export completes loses this run; use a new runtime and start again. Do not rerun the
training cell after a successful export. Keep the complete exported folder.

This notebook has schema, syntax and lightweight logic checks; actual AraT5v2/T4
training, timing, memory use, Drive upload and quality remain to be verified in Colab.


## 1. Settings, local paths and pinned installation


In [ ]:
import os
import sys
import subprocess
import uuid
from datetime import datetime, timezone
from pathlib import Path

# Configure before importing torch, transformers or accelerate.
if any(name in sys.modules for name in ('torch', 'transformers', 'accelerate')):
    raise RuntimeError('Start a fresh runtime; do not reuse an initialized training session.')
os.environ['ACCELERATE_MIXED_PRECISION'] = 'no'
os.environ['ACCELERATE_USE_CPU'] = 'false'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'

MAX_STEPS = 200
TRAIN_BUDGET_MINUTES = 90.0
GRADIENT_ACCUMULATION_STEPS = 8
TEST_SAMPLES = 64
SEED = 42
SOURCE_LENGTH, TARGET_LENGTH = 1024, 128
DRIVE_FREE_GB = None  # Optional: enter current available GB from the Drive storage page.
LAUNCH_GRADIO = False  # Enable before Run all, or run the last cell separately later.

assert type(MAX_STEPS) is int and MAX_STEPS >= 3
assert type(GRADIENT_ACCUMULATION_STEPS) is int and GRADIENT_ACCUMULATION_STEPS > 0
assert type(TEST_SAMPLES) is int and TEST_SAMPLES >= 3
assert 0 < TRAIN_BUDGET_MINUTES <= 180
RUN_NAME = 'single-fp32-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:6]
LOCAL_RUN = Path('/content/arabic-news-single') / RUN_NAME
LOCAL_RUN.mkdir(parents=True, exist_ok=False)
for key, path in {
    'HF_HOME': LOCAL_RUN / 'hf',
    'HF_HUB_CACHE': LOCAL_RUN / 'hf' / 'hub',
    'HF_DATASETS_CACHE': LOCAL_RUN / 'datasets',
    'XDG_CACHE_HOME': LOCAL_RUN / 'cache',
    'TORCH_HOME': LOCAL_RUN / 'torch',
    'TMPDIR': LOCAL_RUN / 'tmp',
}.items():
    path.mkdir(parents=True, exist_ok=True)
    os.environ[key] = str(path)

# Deliberately omit torch: preserve Colab's working CUDA installation.
PINS = [
    'transformers==4.57.1', 'accelerate==1.11.0', 'datasets==4.3.0',
    'huggingface-hub==0.36.0', 'sentencepiece==0.2.1', 'protobuf==5.29.5',
    'safetensors==0.6.2', 'rouge-score==0.1.2', 'trafilatura==2.0.0',
    'urllib3==2.5.0', 'gradio==5.49.1',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *PINS], check=True)
print('Local training directory:', LOCAL_RUN)


## 2. Load stable project helpers and verify T4 / FP32

The source SHA below is the `main` commit used as this notebook's base, with the
verified numerical guards. It supplies data cleaning, tokenization, Arabic ROUGE,
inference and the existing app. This notebook does not import the old training or
checkpoint pipeline. No project dependency install can replace CUDA PyTorch.


In [ ]:
import json
import math
import time
from types import SimpleNamespace
import importlib.metadata as metadata
import torch
import transformers
from accelerate import Accelerator
from huggingface_hub import HfApi
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainingArguments, TrainerCallback, set_seed

SOURCE_COMMIT = 'd1b24cbeef87aa61e45693fd5f7895a9f8db6f05'
REPO_DIR = LOCAL_RUN / 'source'
subprocess.run(['git', 'clone', '--quiet', '--no-checkout',
                'https://github.com/nada-ali-13/Arabic-news-summarizer.git', str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--quiet', '--detach', SOURCE_COMMIT], check=True)
sys.path[:0] = [str(REPO_DIR / 'src'), str(REPO_DIR)]
from arabic_summarizer.data import load_official, fixed_subset, tokenize
from arabic_summarizer.metrics import rouge
from arabic_summarizer.text import normalize
from arabic_summarizer.numerics import ValidatedSeq2SeqCollator, checked_trainer_class


In [ ]:
assert transformers.__version__ == '4.57.1'
assert metadata.version('accelerate') == '1.11.0'
torch_version = tuple(int(v) for v in torch.__version__.split('+')[0].split('.')[:2])
if not (torch_version >= (2, 6) and torch_version < (3, 0)):
    raise RuntimeError('Use a current Colab runtime with torch >=2.6,<3; keep its CUDA build.')
if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
    raise RuntimeError('Select a T4 GPU in Runtime → Change runtime type.')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
set_seed(SEED)
precision_probe = Accelerator(mixed_precision='no')
if (precision_probe.mixed_precision != 'no' or precision_probe.native_amp
        or precision_probe.scaler is not None):
    raise RuntimeError('Stale mixed precision detected. Disconnect and delete the runtime.')
print({'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__,
       'transformers': transformers.__version__, 'precision': precision_probe.mixed_precision,
       'VRAM_GiB': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)})
MODEL_ID, DATASET_ID = 'UBC-NLP/AraT5v2-base-1024', 'csebuetnlp/xlsum'
api = HfApi()
MODEL_REVISION = api.model_info(MODEL_ID).sha
DATASET_REVISION = api.dataset_info(DATASET_ID).sha
data_config = SimpleNamespace(dataset_id=DATASET_ID, dataset_revision=DATASET_REVISION,
                              source_length=SOURCE_LENGTH, target_length=TARGET_LENGTH)
print({'model': MODEL_ID, 'model_revision': MODEL_REVISION,
       'dataset': DATASET_ID, 'dataset_revision': DATASET_REVISION})


## 3. Arabic XL-Sum only; official splits and reproducible preprocessing

Download only the official Arabic v2 archive from `csebuetnlp/xlsum`; this works
with datasets 4.3 without executing the obsolete dataset loading script. Preserve
official train/validation/test membership. Existing cleaning removes empty rows
and exact source/ID/URL overlap, with test > validation > train priority. All
splits undergo conservative Arabic normalization. The test subset is seeded and
is never used for training, validation selection or tuning. Validation is retained
for the overlap audit; no generation validation runs during training.


In [ ]:
dataset = load_official(data_config, LOCAL_RUN / 'arabic-data')
print('Clean official splits:', {name: len(rows) for name, rows in dataset.items()})
test_subset = fixed_subset(dataset['test'], TEST_SAMPLES, SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_fast=False)
if tokenizer.pad_token_id is None or tokenizer.eos_token_id is None:
    raise RuntimeError('Tokenizer needs pad and EOS tokens.')
train_dataset = tokenize(dataset['train'].shuffle(seed=SEED), tokenizer, data_config)
assert len(train_dataset) > 0 and all('labels' in train_dataset[i] for i in range(3))
print({'train_rows': len(train_dataset), 'held_out_test_rows': len(test_subset),
       'tokenizer': type(tokenizer).__name__, 'vocab': len(tokenizer),
       'source_limit': SOURCE_LENGTH, 'target_limit': TARGET_LENGTH})


## 4. Full-parameter model in FP32; inspect supervised padding


In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=torch.float32,
    weights_only=True, low_cpu_mem_usage=True,
).to('cuda')
model.requires_grad_(True)
model.config.use_cache = False
assert all(p.dtype == torch.float32 and p.requires_grad for p in model.parameters())
assert model.config.model_type == 't5' and model.config.vocab_size >= len(tokenizer)
collator = ValidatedSeq2SeqCollator(tokenizer, model, pad_to_multiple_of=8)
batch = collator([train_dataset[0], train_dataset[1]])
assert (batch['labels'] != -100).any(dim=1).all()
assert not (batch['labels'] == tokenizer.pad_token_id).any()
print({'parameters': model.num_parameters(), 'trainable_parameters': model.num_parameters(only_trainable=True),
       'dtype': str(next(model.parameters()).dtype),
       'supervised_tokens': (batch['labels'] != -100).sum(dim=1).tolist(),
       'label_padding': -100})
del batch


## 5. Mount Drive and check space for the final export only

Open the Drive storage page and read **available** account storage (not total
capacity). Enter that number in GB when prompted, or set `DRIVE_FREE_GB` in cell 1.
Mounted Drive disk-usage reports are unreliable, so this uses your current reading
and the model tensor size plus a margin. No quota API, checkpoint reserve, Trash
management, deletion or cleanup runs. The FP32 weights are expected around 1.5 GB;
the actual size is printed after export. Keep some extra space available until then.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/ArabicNewsSummarizer')
EXPORT_DIR = DRIVE_ROOT / RUN_NAME / 'model'
if EXPORT_DIR.parent.exists():
    raise FileExistsError('Export directory already exists. Start a new runtime for a new experiment.')
estimated_weight_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
required_gb = estimated_weight_bytes * 1.25 / 1e9 + 0.1
if DRIVE_FREE_GB is None:
    DRIVE_FREE_GB = float(input('Available Google Drive storage in GB, from the Drive storage page: '))
if not math.isfinite(DRIVE_FREE_GB) or DRIVE_FREE_GB < required_gb:
    raise RuntimeError(f'Final export needs approximately {required_gb:.2f} GB free including margin. No files were deleted.')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Estimated FP32 weights: {estimated_weight_bytes / 1e9:.2f} GB; free-space minimum with margin: {required_gb:.2f} GB')
print('Final export destination:', EXPORT_DIR)


## 6. One Trainer, no saves or evaluations during training

FP32, batch 1, AdamW, gradient accumulation, gradient checkpointing, 6% linear
warmup and clipping at 1.0. The existing numerical guard stops unsafe loss/gradient
updates with local diagnostics. A numerical failure requires investigation; it is
not reported as a successful training run. The budget is a separate, graceful stop.
Timing includes data preparation on the GPU and optimizer work in this same run;
three steps are an approximate estimate, and article lengths can change speed.


In [ ]:
class TimingBudget(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        torch.cuda.synchronize()
        self.started = time.monotonic()
        self.reason = 'max_steps'
        return control

    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.synchronize()
        elapsed = time.monotonic() - self.started
        seconds_per_step = elapsed / state.global_step
        if state.global_step == 3:
            requested_minutes = seconds_per_step * max(0, args.max_steps - state.global_step) / 60
            budget_minutes = max(0, TRAIN_BUDGET_MINUTES - elapsed / 60)
            print(f'\nGPU timing: {seconds_per_step:.1f} seconds / optimizer step (first 3 steps).', flush=True)
            print(f'Estimated remaining to MAX_STEPS: {requested_minutes:.1f} minutes. '
                  f'Remaining training budget: {budget_minutes:.1f} minutes. '
                  f'Expected training remaining: about {min(requested_minutes, budget_minutes):.1f} minutes, '
                  'then final export. This estimate excludes export/evaluation.', flush=True)
        if elapsed >= TRAIN_BUDGET_MINUTES * 60:
            self.reason = 'time_budget'
            print(f'\nTraining budget reached at step {state.global_step}; stopping normally and exporting.', flush=True)
            control.should_training_stop = True
        control.should_save = False
        return control

timing = TimingBudget()
args = Seq2SeqTrainingArguments(
    output_dir=str(LOCAL_RUN / 'trainer'),
    max_steps=MAX_STEPS, learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='adamw_torch', warmup_ratio=0.06, weight_decay=0.01, max_grad_norm=1.0,
    lr_scheduler_type='linear', fp16=False, bf16=False, tf32=False,
    save_strategy='no', eval_strategy='no', predict_with_generate=False,
    load_best_model_at_end=False, report_to='none', logging_steps=10,
    logging_nan_inf_filter=False, seed=SEED, data_seed=SEED,
    dataloader_num_workers=0,
)
Trainer = checked_trainer_class()
trainer = Trainer(model=model, args=args, train_dataset=train_dataset,
                  data_collator=collator, processing_class=tokenizer, callbacks=[timing])
assert trainer.accelerator.mixed_precision == 'no'
assert not trainer.accelerator.native_amp and trainer.accelerator.scaler is None
print('One FP32 experiment ready; timing estimate will print after 3 optimizer steps.')


## 7. Train once → immediately export to Drive

Keep this cell running through **FINAL EXPORT COMPLETE**. It trains once and goes
directly to `save_pretrained()` on Drive, before any generation evaluation or app
work. Never manually interrupt at the time budget: the callback handles it normally.
The export contains model safetensors, model/generation configs, tokenizer and
minimal inference metadata. No Trainer save method or training state is exported.
If Drive upload fails while this runtime is still alive, fix access/space and rerun
this cell: the in-memory `TRAIN_FINISHED` flag skips training and retries only export.
If the runtime disconnects, that flag and the trained weights are lost.


In [ ]:
if not globals().get('TRAIN_FINISHED', False):
    train_result = trainer.train()  # Exactly one real experiment, no resume.
    TRAIN_FINISHED = True

model.gradient_checkpointing_disable()
model.config.use_cache = True
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(EXPORT_DIR, safe_serialization=True, max_shard_size='2GB')
tokenizer.save_pretrained(EXPORT_DIR)
inference_config = {
    'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
    'dataset_id': DATASET_ID, 'dataset_revision': DATASET_REVISION,
    'source_commit': SOURCE_COMMIT, 'source_length': SOURCE_LENGTH,
    'target_length': TARGET_LENGTH, 'seed': SEED,
    'precision': 'float32', 'learning_rate': 5e-5,
    'completed_steps': trainer.state.global_step, 'stop_reason': timing.reason,
    'max_steps_requested': MAX_STEPS, 'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
    'test_samples': len(test_subset),
    'versions': {name: metadata.version(name) for name in ('torch', 'transformers', 'accelerate', 'datasets')},
}
(EXPORT_DIR / 'inference_config.json').write_text(
    json.dumps(inference_config, ensure_ascii=False, indent=2), encoding='utf-8')
files = [p for p in EXPORT_DIR.rglob('*') if p.is_file()]
assert any(p.suffix == '.safetensors' for p in files)
assert not any(p.name in {'optimizer.pt', 'scheduler.pt', 'trainer_state.json', 'training_args.bin'}
               or p.name.startswith('rng_state') for p in files)
assert not list((LOCAL_RUN / 'trainer').glob('checkpoint-*'))
EXPORT_COMPLETE = True
print('FINAL EXPORT COMPLETE:', EXPORT_DIR)
print(f'Actual export size: {sum(p.stat().st_size for p in files) / 1e9:.3f} GB')
print('Files:', [p.name for p in files])
print('Completed optimizer steps:', trainer.state.global_step, '| stop:', timing.reason)


## 8. Independently reload the saved export; held-out Arabic ROUGE

This cell starts only after export. It releases the Trainer/optimizer, then loads
the exported model and tokenizer independently with the standard Auto classes.
Evaluate the seeded official test subset once with deterministic four-beam
generation, using truncated 1,024-token sources and full normalized reference
summaries. ROUGE is mean per-example F1 ×100 with the project's Unicode Arabic
tokenizer, no stemming, and sentence boundaries for ROUGE-Lsum. A subset score
does not represent the full test set, and overlap scores do not establish factuality.
Predictions, references, IDs and scores stay local; download the JSON if desired.


In [ ]:
import gc
assert EXPORT_COMPLETE
del trainer, model, collator
gc.collect()
torch.cuda.empty_cache()
tokenizer = AutoTokenizer.from_pretrained(EXPORT_DIR, use_fast=False, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    EXPORT_DIR, dtype=torch.float32, local_files_only=True,
    use_safetensors=True, weights_only=True,
).to('cuda').eval()
model.config.use_cache = True

def generate_summary(text):
    inputs = tokenizer(normalize(text), max_length=SOURCE_LENGTH, truncation=True, return_tensors='pt')
    inputs = {key: value.to('cuda') for key, value in inputs.items()}
    with torch.inference_mode():
        ids = model.generate(**inputs, max_new_tokens=TARGET_LENGTH, min_new_tokens=8,
                             num_beams=4, do_sample=False, no_repeat_ngram_size=3,
                             repetition_penalty=1.1, length_penalty=1.0, early_stopping=True)
    return normalize(tokenizer.decode(ids[0], skip_special_tokens=True))

from tqdm.auto import tqdm
examples = []
for row in tqdm(test_subset, desc='Held-out Arabic test generation'):
    examples.append({'id': row['id'], 'source': row['text'], 'reference': row['summary'],
                     'prediction': generate_summary(row['text'])})
scores = rouge([r['prediction'] for r in examples], [r['reference'] for r in examples])
evaluation_path = LOCAL_RUN / 'held_out_rouge.json'
evaluation_path.write_text(json.dumps({'metrics': scores, 'examples': examples,
                                      'config': inference_config}, ensure_ascii=False, indent=2), encoding='utf-8')
print({'test_rows': len(examples), 'seed': SEED, 'mean_F1_times_100': scores})
print('Local evaluation JSON:', evaluation_path)


## 9. Review Arabic articles, full references and generated summaries


In [ ]:
from IPython.display import HTML, display
from html import escape
for row in examples[:3]:
    display(HTML('<div dir="rtl" style="text-align:right;line-height:1.9">'
                 '<h3>المقال</h3><p>' + escape(row['source']) + '</p>'
                 '<h3>الملخص المرجعي</h3><p>' + escape(row['reference']) + '</p>'
                 '<h3>ملخص النموذج</h3><p>' + escape(row['prediction']) + '</p></div>'))


## 10. Try custom Arabic text using the existing inference logic

Edit the text and rerun this cell. This uses the project's long-document handling
and short/medium/long preferences; evaluation above uses the documented truncated
input protocol. The already reloaded model remains FP32 for both custom text and UI.


In [ ]:
from arabic_summarizer.inference import Summarizer
summarizer = Summarizer(str(EXPORT_DIR), device='cuda', source_length=SOURCE_LENGTH,
                        model=model, tokenizer=tokenizer)
CUSTOM_TEXT = ('أعلنت الجامعة افتتاح مكتبة جديدة تضم قاعات للمطالعة ومختبرًا رقميًا. '
               'وقالت إدارة الجامعة إن المكتبة ستفتح أبوابها للطلاب الأسبوع المقبل، '
               'وستوفر مصادر إلكترونية وبرامج تدريبية لمساعدة الباحثين في الوصول إلى المعلومات.')
print(summarizer.summarize(CUSTOM_TEXT, preference='متوسط'))


## 11. Optional existing Arabic Gradio app and later loading

Set `LAUNCH_GRADIO=True` and run the next cell to open the app in Colab. `share=True`
creates a temporary public Gradio URL; anyone with that link can use it while the
runtime runs. Text input works directly; some article URLs may block extraction.

In a later fresh Colab runtime, mount Drive and install dependencies as above,
then set `ARABIC_SUMMARIZER_MODEL` to the printed `.../single-fp32-.../model`
folder. Run `python app.py --share` from a project checkout (with `PYTHONPATH=src`).
For a local computer, copy the **whole** model folder, install the project, and run:

```bash
export ARABIC_SUMMARIZER_MODEL=/absolute/path/to/single-fp32-.../model
python app.py
```

Independent loading needs only `AutoTokenizer.from_pretrained(saved_folder,
use_fast=False)` and `AutoModelForSeq2SeqLM.from_pretrained(saved_folder,
dtype=torch.float32)`. The existing app automatically uses FP32 on CPU and FP16
on CUDA when it loads a model itself; the model injected into this notebook's app
stays FP32. FP16 **training** is never enabled here. No old run files are needed.


In [ ]:
if LAUNCH_GRADIO:
    from app import build_app
    demo = build_app(summarizer=summarizer)
    demo.launch(share=True, inline=True)
else:
    print('Optional UI skipped. Set LAUNCH_GRADIO=True and rerun this cell to launch.')
